# A_03 | Coorte final e dataset supervisionado

Fluxo único e oficial: **auditar → gerar candidata → publicar → gerar artefato (partição 70/15/15 em blocos)**.

Substitui o antigo `A_03_coorte_protocolo` (histórico em `notebooks/A/historico/`).
Não usa `src.protocolo.verificacoes`. Contrato: `configs/final_cohort_v1.yaml`. Partição: `configs/supervised.yaml` (`temporal`).

Partição (anos 2019, 2023–2025): `train_ref` 0,70 · `val_ref` 0,15 · `test_ref` 0,15. Anos 2020–2022 → `test_pandemic`; 2026 → `exploracao`. Testes seguem fechados até o G9.

In [7]:
import json, shutil, sys, os
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "configs" / "supervised.yaml").exists())
os.chdir(ROOT); sys.path.insert(0, str(ROOT))
for m in [m for m in sys.modules if m == "src" or m.startswith("src.")]:
    del sys.modules[m]  # evita 'src' antigo em cache do kernel

import pandas as pd
from src.data.final_cohort_cli import main as cohort_cli
from src.data.supervised_artifact import gerar_artefato_supervisionado

CAND = ROOT / "outputs/coorte_candidata"
PUB = ROOT / "data/analytical/coorte_final.parquet"

# Interruptores: rode tudo na 1ª vez; depois desligue o que já está pronto.
GERAR_CANDIDATA = True
PUBLICAR = True
GERAR_ARTEFATO = True

# Contagens de 24/10 (referência, não é regra): divergência pode vir de regras posteriores da coorte.
REF_24_10 = {"train_ref": 556_462, "val_ref": 119_243, "test_ref": 119_245, "test_pandemic": 2_915_829}
print("Raiz:", ROOT)

Raiz: c:\Users\eduar\Documents\GitHub\Projeto_Integrador


## 1. Auditoria (não remove nada)

In [8]:
if GERAR_CANDIDATA:
    assert cohort_cli(["--mode", "audit", "--overwrite"]) == 0
    display(pd.read_csv(CAND / "resumo_regras.csv"))

Fonte: C:\Users\eduar\Documents\GitHub\Projeto_Integrador\data\processed\dados-v2\srag.parquet
Contrato: C:\Users\eduar\Documents\GitHub\Projeto_Integrador\configs\final_cohort_v1.yaml
Colunas da auditoria: ['hospital', 'dt_interna', 'dt_sin_pri', 'evolucao']
Linhas carregadas: 4666910

Auditoria concluída.
Regras aprovadas para exclusão: {'hospital_nao_confirmado': 232253, 'dt_interna_invalida': 304245, 'evolucao_fora_alvo_principal': 530370, 'dt_interna_antes_periodo': 48, 'dt_interna_apos_periodo': 76, 'intervalo_sintomas_negativo': 24361, 'intervalo_sintomas_maior_limite': 11708}
Regras pendentes de decisão: {}
Diretório dos resultados: C:\Users\eduar\Documents\GitHub\Projeto_Integrador\outputs\coorte_candidata
Nenhuma exclusão foi aplicada.
Nenhuma candidata foi gerada.
Nenhuma coorte foi publicada.


,regra,linhas_sinalizadas,total_linhas,proporcao_fonte
0,hospital_nao_confirmado,232253,4666910,0.049766
1,dt_interna_invalida,304245,4666910,0.065192
2,evolucao_fora_alvo_principal,530370,4666910,0.113645
3,dt_interna_antes_periodo,48,4666910,0.000010
4,dt_interna_apos_periodo,76,4666910,0.000016
5,intervalo_sintomas_negativo,24361,4666910,0.005220
6,intervalo_sintomas_maior_limite,11708,4666910,0.002509


## 2. Candidata + validação
Gera `outputs/coorte_candidata/`. O waterfall precisa fechar (fonte = excluídas + candidata).

In [9]:
if GERAR_CANDIDATA:
    assert cohort_cli(["--mode", "candidate", "--overwrite"]) == 0
val = json.loads((CAND / "validacao_publicacao.json").read_text(encoding="utf-8"))
rec = val["population_reconciliation"]
print(rec)
assert rec["closed"], "waterfall não fecha: não publique"
display(pd.read_csv(CAND / "coorte_waterfall.csv"))

Fonte: C:\Users\eduar\Documents\GitHub\Projeto_Integrador\data\processed\dados-v2\srag.parquet
Contrato: C:\Users\eduar\Documents\GitHub\Projeto_Integrador\configs\final_cohort_v1.yaml
Row groups: 28
Linhas da fonte: 4666910
Row group 1/28: fonte=200000, candidata=155294
Row group 2/28: fonte=20941, candidata=16240
Row group 3/28: fonte=200000, candidata=171843
Row group 4/28: fonte=136420, candidata=117146
Row group 5/28: fonte=200000, candidata=168924
Row group 6/28: fonte=67986, candidata=57355
Row group 7/28: fonte=200000, candidata=169011
Row group 8/28: fonte=79453, candidata=67269
Row group 9/28: fonte=200000, candidata=163212
Row group 10/28: fonte=200000, candidata=163782
Row group 11/28: fonte=160577, candidata=131240
Row group 12/28: fonte=200000, candidata=165982
Row group 13/28: fonte=200000, candidata=165926
Row group 14/28: fonte=200000, candidata=166236
Row group 15/28: fonte=200000, candidata=165986
Row group 16/28: fonte=200000, candidata=166027
Row group 17/28: fonte

,ordem,regra,excluidas_na_etapa,restantes_apos_etapa
0,1,hospital_nao_confirmado,232253,4434657
1,2,dt_interna_invalida,72016,4362641
2,3,evolucao_fora_alvo_principal,447436,3915205
3,4,dt_interna_antes_periodo,29,3915176
4,5,dt_interna_apos_periodo,34,3915142
5,6,intervalo_sintomas_negativo,20918,3894224
6,7,intervalo_sintomas_maior_limite,9496,3884728


## 3. Publicar
Copia a candidata para `data/analytical/coorte_final.parquet` (com backup). O CLI não tem modo publish.

In [10]:
if PUBLICAR:
    PUB.parent.mkdir(parents=True, exist_ok=True)
    if PUB.exists():
        shutil.copy2(PUB, PUB.with_suffix(".parquet.bak"))
    shutil.copy2(CAND / "coorte_final.parquet", PUB)

    manifest_path = PUB.with_name("coorte_final.manifest.json")
    shutil.copy2(CAND / "manifesto_coorte.json", manifest_path)

    # Marca a publicação e a revisão de paridade com o legado
    m = json.loads(manifest_path.read_text(encoding="utf-8"))
    m["publication_performed"] = True
    m["legacy_parity_reviewed"] = True
    m["provisional_comparison_reviewed"] = True
    manifest_path.write_text(json.dumps(m, indent=2, ensure_ascii=False), encoding="utf-8")

print("Publicada:", PUB, f"{PUB.stat().st_size/1e6:.1f} MB")

Publicada: c:\Users\eduar\Documents\GitHub\Projeto_Integrador\data\analytical\coorte_final.parquet 255.0 MB


## 4. Dataset supervisionado + partição-mestre
Grava `dataset_supervisionado.parquet`, `partitions_master.parquet` e `manifesto_particao.json`. Determinístico (hash do `group_id`, seed 42). Leva alguns minutos.

In [11]:
if GERAR_ARTEFATO:
    res = gerar_artefato_supervisionado(ROOT, cohort_kind="final", exportar_particao=True, overwrite=True)
    contagens = res["partition_counts"]
    tab = pd.DataFrame({"linhas": pd.Series(contagens), "ref_24_10": pd.Series(REF_24_10)})
    tab["igual_ref"] = tab["linhas"] == tab["ref_24_10"]
    display(tab)

,linhas,ref_24_10,igual_ref
test_pandemic,2915829,2915829,True
test_ref,119245,119245,True
train_ref,556462,556462,True
val_ref,119243,119243,True


## 5. Conferência da razão 70/15/15
Percentuais só sobre os blocos dos anos compartilhados (os pandêmicos e 2026 ficam de fora por desenho).

In [12]:
if GERAR_ARTEFATO:
    comp = {k: v for k, v in contagens.items() if k in ("train_ref", "val_ref", "test_ref")}
    pct = pd.Series(comp) / sum(comp.values())
    esperado = pd.Series({"train_ref": .70, "val_ref": .15, "test_ref": .15})
    display(pd.DataFrame({"pct": pct.round(4), "esperado": esperado}))
    assert ((pct - esperado).abs() < 0.005).all(), "razão fora de 70/15/15 (tolerância 0,5 p.p.)"
    print("OK: 70/15/15. Testes seguem fechados até o G9.")

,pct,esperado
test_ref,0.15,0.15
train_ref,0.70,0.70
val_ref,0.15,0.15


OK: 70/15/15. Testes seguem fechados até o G9.
